# 1 amending src files

In [265]:
FDIR = 'modf_rnt_wrk_div'

In [266]:
import pandas as pd

def ld(p):
    return pd.read_csv(p, sep=',')

In [267]:
def rm_col(ncols, c):
    if c in ncols:
        ncols.remove(c)
    return ncols

In [268]:
IOU_COLS_GDA = ["3_test_GDA_iou", "3_test/GDA/iou", "1_test/GDA/iou",  
                "2_test/GDA/iou", "1_test_GDA_iou", "2_test_GDA_iou"]
IOU_COLS_SYNT = ["3_test_SYNT_iou", "1_test/SYNT/iou", "2_test/SYNT/iou", 
            "3_test/SYNT/iou", "1_test_SYNT_iou", "2_test_SYNT_iou"]
IOU_COLS_DK = ["3_test_DK_iou", "1_test/DK/iou", "2_test/DK/iou", 
            "3_test/DK/iou", "1_test_DK_iou", "2_test_DK_iou"]

IOU_COLS = IOU_COLS_GDA + IOU_COLS_SYNT + IOU_COLS_DK
# SPLIT_COLUMNS = ['val', 'Sweep']
SCOLS = ["batch_size", "epochs", "warmup_epochs", "wd", "ema", "sub", 
        "ID", "loss", "lrdec", "lrenc", "mod_ph1", "mod_ph2", "Sweep",
        "val", "1_epoch", "2_epoch", "3_epoch", 'Runtime'] + IOU_COLS

def ld_scols(p):
    df = pd.read_csv(p, sep=',')
    cols = df.columns
    cols = list(set(cols) & set(SCOLS))
    ncols = list(set(cols) & set(SCOLS))
    df = df[df['State'] == 'finished']
    # if 'ID' in cols:
    #     df.set_index('ID', inplace=True, drop=False)
        # df.reindex('ID')
    if 'ph3' in p:
        ncols = rm_col(ncols, "2_test/DK/iou")
        ncols = rm_col(ncols, "2_test/SYNT/iou")
        ncols = rm_col(ncols, "2_test/GDA/iou")
        ncols = rm_col(ncols, "2_test_DK_iou")
        ncols = rm_col(ncols, "2_test_GDA_iou")
        ncols = rm_col(ncols, "2_test_SYNT_iou")
    if 'S' in p:
        ncols = rm_col(ncols, "1_test_DK_iou")
        ncols = rm_col(ncols, "1_test_GDA_iou")
        ncols = rm_col(ncols, "1_test_SYNT_iou")
    for col in cols:
        if '/' in col:
            # print('sign found in', col)
            df[col.replace('/', '_')] = df[col]
            df = df.drop(columns=[col])
            if col in ncols:
                ncols.remove(col)
            ncols.append(col.replace('/', '_'))
    # print(ncols)
    return df[ncols].fillna({'ema': False})

In [269]:
def drop_empty_cols(df: pd.DataFrame):
    return df.dropna(axis='columns')

In [270]:
def map_dict_val(d: dict, val, deff):
    for k, v in d.items():
        # print(k, v)
        if k in val:
            # print(len(val.split('_')))
            if ('_' in val and len(val.split('_')) > 3) or 'ph1_gda' in val:
                return 'gda'
            return v
    if 'ph1_gda' in val:
        return 'gda'
    return deff

In [271]:
fn_tr = {
    'ph2': 'subm',
    'ph3': 'dk',
    'dk': 'dk',
    'subs': 'sub',
    'S': 'sub',
}

def add_tr(df, fn):
    df['train'] = map_dict_val(fn_tr, fn, 's')
    return df

In [272]:
from numpy import select

val_val = {
    "/users/project1/pt01299/synt/gda70/train/index_val.csv": "gda",
    "/users/project1/pt01299/synt/gda70/train/index_val_val.csv": "gda",
    "/users/project1/pt01299/synt/gda70/train/index_val_tr.csv": "gda",
    "/users/project1/pt01299/synt/segformer_dataset255_all/val/index.csv": "s",
    "/users/project1/pt01299/synt/mix_val.csv": "m",
    "/users/project1/pt01299/synt/DK/osfstorage/dataset_v2/solardk_dataset_neurips_v2/gentofte_trainval/val/index.csv": "dk"
}

def map_val(df):
    if 'val' not in df.columns:
        df['val'] = list(val_val.keys())[0]
    v = df['val'].astype(str)
    conds = [v.str.contains(k, regex=False, na=False) for k in val_val.keys()]
    choices = list(val_val.values())
    df['val'] = select(conds, choices, default='s')
    return df

In [273]:
def add_tr_val(df):
    df['tr_val'] = df['train'] + '_' + df['val']
    return df

In [274]:
def add_src(df, fn):
    sp = fn.split('_')
    if len(sp) > 2:
        df['src'] = sp[1] + '_' + sp[2].replace('.csv', '')
    elif 'm' in fn:
        df['src'] = 'subm'
    else:
        df['src'] = 'x'
    df['fn'] = fn
    return df

In [275]:
# def add_workload(df: pd.DataFrame, fn):
#     def sub_mult(x):
#         if pd.isna(x):
#             return 1
#         if x == 'mix':
#             return 1/8
#         if x == 'composite':
#             return 1/9
#         return float(x) / 100
#     df['workload'] = df['train'].map(DS_SIZES_TR) * df['sub'].apply(sub_mult) * df['train'].map(DS_SIZES_TR) # factor
#     df['workload'] += df['val'].map(DS_SIZES_VAL) # subs, m, ...

In [276]:
DS_SIZES_TR = { # TODO
    's': 12353*0.7,
    'sub': 12353*0.7, # s, later mult
    'dk': 755,
    'gda': 18,
}

DS_SIZES_VAL = { # TODO
    's': 12353*0.3,
    'sub': 12353*0.3, # s, later mult
    'dk': 123,
    'gda': 20,
}

S_SUB_S_FCT = { # TODO
    'mix': 100/9,
    'composite': 100/8
}

DS_W_MULT = {
    's': 0.1,
    'sub': 0.1,
    'gda': 1,
    'dk': 1,
    # 'm': 0.5,
    # 'subm': 0.25,
}

DS_DOM = {
    's': 0.75,
    'sub': 0.75,
    'gda': 1,
    'dk': 0,
    # 'm': 0.25, # in fact also subm; 
    # 'subm': 0.5,
}
DS_DOM = {
    's': 0.25,
    'sub': 0.25,
    'gda': 1,
    'dk': -1,
}

DS_REAL = {
    's': 0,
    'sub': 0,
    'gda': 1,
    'dk': 1,
    # 'm': 0.5,
    # 'subm': 0.25,
}
DS_REAL = {
    's': -1,
    'sub': -1,
    'gda': 1,
    'dk': 1,
}

In [277]:
# def add_workload(df: pd.DataFrame, fn):
#     # su multiplier
#     su_map = df['sub'].map(S_SUB_S_FCT) # mix
#     df['sub_mult'] = su_map.fillna(df['sub'] / 100) # numb
#     df['sub_mult'] = df['sub_mult'].fillna(1) # NaN - no sub

#     # t-dependent part
#     w_t_base = df['train'].map(DS_W_MULT) * df['train'].map(DS_SIZES_TR) * df['sub_mult']

#     w_t_m = (
#         DS_W_MULT['s'] * DS_SIZES_TR['s'] * df['sub_mult'] +
#         DS_W_MULT['dk'] * DS_SIZES_TR['dk'] * df['sub_mult']
#     )

#     df['w_t'] = w_t_base.where(df['train'] != 'm', w_t_m)

#     # v-dependent part
#     df['w_v'] = df['val'].map(DS_W_MULT) * df['v'].map(DS_SIZES_TR)

#     # final
#     df['workload'] = df['w_t'] + df['w_v']


In [278]:
def add_sub_sumlt(df: pd.DataFrame) -> pd.DataFrame:
    sub_map = df['sub'].map(S_SUB_S_FCT) # mix
    df['sub_mult'] = sub_map.fillna(df['sub']) # numb
    df['sub_mult'] = df['sub_mult'].fillna(100) # NaN - no sub
    df['sub_mult'] = df['sub_mult'].astype(float) / 100

    return df

In [279]:
def add_workload(df: pd.DataFrame) -> pd.DataFrame:
    # train
    msk = df['train'].isin(['m', 'subm'])
    w_t_base = df['train'].map(DS_W_MULT) * df['train'].map(DS_SIZES_TR) * df['sub_mult']
    w_t_m = (
        DS_W_MULT['s'] * DS_SIZES_TR['s'] * df['sub_mult'] +
        DS_W_MULT['dk'] * DS_SIZES_TR['dk']    )
    df['w_t'] = w_t_base.where(~msk, w_t_m)

    # val
    # df['w_v'] = df['val'].map(DS_W_MULT) * df['val'].map(DS_SIZES_TR)
    msk = df['val'].isin(['m', 'subm'])
    w_v_base = df['val'].map(DS_W_MULT) * df['val'].map(DS_SIZES_VAL) * df['sub_mult']
    w_v_m = (
        DS_W_MULT['s'] * DS_SIZES_VAL['s'] * df['sub_mult'] +
        DS_W_MULT['dk'] * DS_SIZES_VAL['dk']
    )
    df['w_v'] = w_v_base.where(~msk, w_v_m)

    # final
    df['workload'] = df['w_t'] + df['w_v']

    return df

In [280]:
def add_diversity(df: pd.DataFrame) -> pd.DataFrame:
    x0, y0 = 0, 0 # ref pt zero
    
    # --- re train ---
    msk = df['train'].isin(['m', 'subm'])
    re_t_base = df['train'].map(DS_REAL)
    re_t_m_num = (
        DS_REAL['s'] * DS_SIZES_TR['s'] * df['sub_mult'] +
        DS_REAL['dk'] * DS_SIZES_TR['dk']
    )
    re_t_m_den = (
        DS_SIZES_TR['s'] * df['sub_mult'] +
        DS_SIZES_TR['dk']
    )
    re_t_m = re_t_m_num / re_t_m_den
    df['re_t'] = re_t_base.where(~msk, re_t_m)

    # --- do train ---
    do_t_base = df['train'].map(DS_DOM)
    do_t_m_num = (
        DS_DOM['s'] * DS_SIZES_TR['s'] * df['sub_mult'] +
        DS_DOM['dk'] * DS_SIZES_TR['dk']
    )
    do_t_m_den = (
        DS_SIZES_TR['s'] * df['sub_mult'] +
        DS_SIZES_TR['dk']
    )
    do_t_m = do_t_m_num / do_t_m_den
    df['do_t'] = do_t_base.where(~msk, do_t_m)

    # --- re val ---
    # df['re_v'] = df['val'].map(DS_REAL)
    msk = df['val'].isin(['m', 'subm'])
    re_v_base = df['val'].map(DS_REAL)
    re_v_m_num = (
        DS_REAL['s'] * DS_SIZES_VAL['s'] * df['sub_mult'] +
        DS_REAL['dk'] * DS_SIZES_VAL['dk']
    )
    re_v_m_den = (
        DS_SIZES_VAL['s'] * df['sub_mult'] +
        DS_SIZES_VAL['dk']
    )
    re_v_m = re_v_m_num / re_v_m_den
    df['re_v'] = re_v_base.where(~msk, re_v_m)

    df['real'] = df['re_t'] + df['re_v']

    # --- do val ---
    # df['do_v'] = df['val'].map(DS_DOM)
    do_v_base = df['val'].map(DS_DOM)
    do_v_m_num = (
        DS_DOM['s'] * DS_SIZES_VAL['s'] * df['sub_mult'] +
        DS_DOM['dk'] * DS_SIZES_VAL['dk']
    )
    do_v_m_den = (
        DS_SIZES_VAL['s'] * df['sub_mult'] +
        DS_SIZES_VAL['dk']
    )
    do_v_m = do_v_m_num / do_v_m_den
    df['do_v'] = do_v_base.where(~msk, do_v_m)

    df['dom'] = df['do_t'] + df['do_v']

    # --- euclidean distance ---
    df['dist'] = ((df['real'] - x0)**2 + (df['dom'] - y0)**2)**0.5

    return df

In [281]:
f = 'dk.csv'
f = 'subs_m.csv'
f = 'ph2_dk_dk.csv'
f = 'ph1_gda.csv'
df = ld_scols(f)
df = add_tr(df, f)
df = map_val(df)
df = add_tr_val(df)
df = add_src(df, f)
df = add_sub_sumlt(df)
df = add_diversity(df)
df = add_workload(df)
if 'ID' not in df.columns:
        df['ID'] = 'keine'
df.head()

,val,wd,lrenc,epochs,Runtime,loss,Sweep,lrdec,1_epoch,sub,...,re_t,do_t,re_v,real,do_v,dom,dist,w_t,w_v,workload
0,dk,0.015,0.00005,32,381,Dice+Focal,afi2hm6i,0.000005,31,NaN,...,1,1.0,1,2,-1.0,0.0,2.000000,18.0,123.0,141.0
1,dk,0.015,0.00005,32,394,Dice+Focal,afi2hm6i,0.000005,31,NaN,...,1,1.0,1,2,-1.0,0.0,2.000000,18.0,123.0,141.0
2,dk,0.030,0.00005,32,200,Dice+Focal,afi2hm6i,0.000005,31,NaN,...,1,1.0,1,2,-1.0,0.0,2.000000,18.0,123.0,141.0
3,dk,0.015,0.00005,32,190,Dice+Focal,afi2hm6i,0.000005,31,NaN,...,1,1.0,1,2,-1.0,0.0,2.000000,18.0,123.0,141.0
4,gda,0.015,0.00005,32,115,Dice+Focal,afi2hm6i,0.000005,31,NaN,...,1,1.0,1,2,1.0,2.0,2.828427,18.0,20.0,38.0


In [282]:
def proc_f(f):
    df = ld_scols(f)
    df = add_tr(df, f)
    df = map_val(df)
    df = add_tr_val(df)
    df = add_src(df, f)
    df = add_sub_sumlt(df)
    df = drop_empty_cols(df)
    df = add_diversity(df)
    df = add_workload(df)
    if 'ID' not in df.columns:
        df['ID'] = 'keine'
        print('no id in', f)
    df.to_csv(f'{FDIR}/{f}')
    
    return df

In [283]:
import os

files = [f for f in os.listdir() if f.lower().endswith(".csv")]
files.remove('da.csv')
files

['30.csv',
 'dk.csv',
 'full.csv',
 'ph1_gda.csv',
 'ph2_dk_dk.csv',
 'ph2_dk_gda.csv',
 'ph2_sub_gda.csv',
 'ph2_sub_s.csv',
 'ph2_s_gda.csv',
 'ph2_s_s.csv',
 'ph3_subm_gda.csv',
 'ph3_subm_gda_gda.csv',
 'ph3_subm_m.csv',
 'ph3_subm_m_gda.csv',
 'ph3_subm_s.csv',
 'ph3_subm_s_gda.csv',
 'S3.csv',
 'S4.csv',
 'short.csv',
 'subs_16.csv',
 'subs_16A.csv',
 'subs_m.csv']

## proces each file

In [284]:
ph1 = pd.DataFrame()
ph2 = pd.DataFrame()
ph3 = pd.DataFrame()

for ff in files:
    print(ff)
    df = proc_f(ff)
    if 'ph2' in ff:
        ph2 = pd.concat([df, ph2], ignore_index=True)
    elif 'ph3' in ff:
        ph3 = pd.concat([df, ph3], ignore_index=True)
    else:
        try:
            if 'S' in ff:
                print(len(df.columns))
                print(len(pd.unique(df.columns)))
                print(df.columns)
            ph1 = pd.concat([df, ph1], ignore_index=True)
        except Exception as e:
            print('ph1', ff, e)

# ph1 = ph1.dropna(axis='columns')
ph2 = ph2.dropna(axis='columns')
ph3 = ph3.dropna(axis='columns')

ph1.to_csv(f'{FDIR}/proc_ph1.csv')
ph2.to_csv(f'{FDIR}/proc_ph2.csv')
ph3.to_csv(f'{FDIR}/proc_ph3.csv')

30.csv
dk.csv
full.csv
ph1_gda.csv
ph2_dk_dk.csv


C:\Users\admin\AppData\Local\Temp\ipykernel_20820\2474272437.py:43: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  return df[ncols].fillna({'ema': False})
C:\Users\admin\AppData\Local\Temp\ipykernel_20820\2474272437.py:43: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  return df[ncols].fillna({'ema': False})


ph2_dk_gda.csv
ph2_sub_gda.csv
ph2_sub_s.csv
ph2_s_gda.csv
ph2_s_s.csv
ph3_subm_gda.csv
ph3_subm_gda_gda.csv
ph3_subm_m.csv
ph3_subm_m_gda.csv
ph3_subm_s.csv
ph3_subm_s_gda.csv
S3.csv
31
31
Index(['val', 'wd', 'lrenc', 'epochs', 'Runtime', 'loss', 'Sweep', 'lrdec',
       '1_epoch', 'sub', 'warmup_epochs', 'batch_size', 'ID',
       '1_test_SYNT_iou', '1_test_GDA_iou', '1_test_DK_iou', 'train', 'tr_val',
       'src', 'fn', 'sub_mult', 're_t', 'do_t', 're_v', 'real', 'do_v', 'dom',
       'dist', 'w_t', 'w_v', 'workload'],
      dtype='object')
S4.csv
29
29
Index(['warmup_epochs', 'batch_size', '1_epoch', 'ID', 'Runtime', 'sub', 'wd',
       'lrenc', 'loss', 'lrdec', '1_test_DK_iou', '1_test_SYNT_iou',
       '1_test_GDA_iou', 'train', 'val', 'tr_val', 'src', 'fn', 'sub_mult',
       're_t', 'do_t', 're_v', 'real', 'do_v', 'dom', 'dist', 'w_t', 'w_v',
       'workload'],
      dtype='object')
short.csv
subs_16.csv
subs_16A.csv
subs_m.csv
no id in subs_m.csv


C:\Users\admin\AppData\Local\Temp\ipykernel_20820\2474272437.py:43: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  return df[ncols].fillna({'ema': False})
C:\Users\admin\AppData\Local\Temp\ipykernel_20820\2474272437.py:43: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  return df[ncols].fillna({'ema': False})


drop duplicate iou cols
drop empty ious
====stack all
====join 1-2 by mod1
====join 2-3 by mod3
====generate comb_key 1-2-3
====grouped agg stats by comb_key
stack disjoint to joint - to załatwia skrypt etl, tutaj tylko prawdziwe i pełne fazy

# 2 joins + save wide

In [285]:
cnc = pd.concat([ph1, ph2, ph3])
cnc.head()

,warmup_epochs,batch_size,1_epoch,loss,epochs,val,Runtime,ema,sub,wd,...,2_epoch,mod_ph1,2_test_GDA_iou,2_test_SYNT_iou,2_test_DK_iou,mod_ph2,3_epoch,3_test_DK_iou,3_test_GDA_iou,3_test_SYNT_iou
0,2,8,19.0,Dice+Focal,20.0,m,4416,True,composite,0.05,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2,8,19.0,Dice+Focal,20.0,s,4085,False,15,0.01,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2,16,19.0,Dice+Focal,20.0,m,5381,True,15,0.03,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,4,16,19.0,Dice+Focal,20.0,gda,4572,False,25,0.03,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2,16,19.0,Dice+Focal,20.0,gda,2996,False,15,0.05,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [286]:
cnc.columns

Index(['warmup_epochs', 'batch_size', '1_epoch', 'loss', 'epochs', 'val',
       'Runtime', 'ema', 'sub', 'wd', 'lrenc', 'Sweep', 'lrdec',
       '1_test_DK_iou', '1_test_SYNT_iou', '1_test_GDA_iou', 'train', 'tr_val',
       'src', 'fn', 'sub_mult', 're_t', 'do_t', 're_v', 'real', 'do_v', 'dom',
       'dist', 'w_t', 'w_v', 'workload', 'ID', '2_epoch', 'mod_ph1',
       '2_test_GDA_iou', '2_test_SYNT_iou', '2_test_DK_iou', 'mod_ph2',
       '3_epoch', '3_test_DK_iou', '3_test_GDA_iou', '3_test_SYNT_iou'],
      dtype='object')

In [287]:
cnc.to_csv(f'{FDIR}/cnc123b.csv')

In [288]:
df12 = pd.merge(ph1, ph2, how='inner', left_on=['ID'], right_on=['mod_ph1'])
df12.head()

,warmup_epochs_x,batch_size_x,1_epoch,loss_x,epochs_x,val_x,Runtime_x,ema_x,sub_x,wd_x,...,re_t_y,do_t_y,re_v_y,real_y,do_v_y,dom_y,dist_y,w_t_y,w_v_y,workload_y
0,6,16,15.0,Dice+Focal,16.0,gda,979,False,15,0.05,...,-0.264156,-0.209903,1.000000,0.735844,1.000000,0.790097,1.079685,884.706500,3.000000,887.706500
1,6,16,15.0,Dice+Focal,16.0,gda,979,False,15,0.05,...,-0.177510,-0.264057,-1.000000,-1.177510,0.250000,-0.014057,1.177593,863.088750,46.323750,909.412500
2,6,16,15.0,Dice+Focal,16.0,gda,979,False,15,0.05,...,-0.264156,-0.209903,1.000000,0.735844,1.000000,0.790097,1.079685,884.706500,3.000000,887.706500
3,6,16,15.0,Dice+Focal,16.0,gda,979,False,15,0.05,...,-0.119938,-0.300039,-0.539986,-0.659925,-0.037509,-0.337547,0.741241,851.078889,164.176667,1015.255556
4,6,16,15.0,Dice+Focal,16.0,gda,979,False,15,0.05,...,-0.264156,-0.209903,1.000000,0.735844,1.000000,0.790097,1.079685,884.706500,3.000000,887.706500


In [289]:
df12.columns

Index(['warmup_epochs_x', 'batch_size_x', '1_epoch', 'loss_x', 'epochs_x',
       'val_x', 'Runtime_x', 'ema_x', 'sub_x', 'wd_x', 'lrenc_x', 'Sweep_x',
       'lrdec_x', '1_test_DK_iou', '1_test_SYNT_iou', '1_test_GDA_iou',
       'train_x', 'tr_val_x', 'src_x', 'fn_x', 'sub_mult_x', 're_t_x',
       'do_t_x', 're_v_x', 'real_x', 'do_v_x', 'dom_x', 'dist_x', 'w_t_x',
       'w_v_x', 'workload_x', 'ID_x', '2_epoch', 'warmup_epochs_y', 'mod_ph1',
       'batch_size_y', 'loss_y', 'ID_y', 'epochs_y', 'val_y', 'Runtime_y',
       'ema_y', 'sub_y', 'wd_y', 'lrenc_y', 'Sweep_y', 'lrdec_y',
       '2_test_GDA_iou', '2_test_SYNT_iou', '2_test_DK_iou', 'train_y',
       'tr_val_y', 'src_y', 'fn_y', 'sub_mult_y', 're_t_y', 'do_t_y', 're_v_y',
       'real_y', 'do_v_y', 'dom_y', 'dist_y', 'w_t_y', 'w_v_y', 'workload_y'],
      dtype='object')

In [290]:
df123 = pd.merge(df12, ph3, how='inner', left_on=['ID_y'], right_on=['mod_ph2'])
df123.head()

,warmup_epochs_x,batch_size_x,1_epoch,loss_x,epochs_x,val_x,Runtime_x,ema_x,sub_x,wd_x,...,re_t,do_t,re_v,real,do_v,dom,dist,w_t,w_v,workload
0,6,8,15.0,Dice+Focal,16.0,gda,632,False,mix,0.05,...,1,1.0,1,2,-1.0,0.0,2.0,18.0,123.0,141.0
1,6,8,15.0,Dice+Focal,16.0,gda,632,False,mix,0.05,...,1,1.0,1,2,-1.0,0.0,2.0,18.0,123.0,141.0
2,6,8,15.0,Dice+Focal,16.0,gda,632,False,mix,0.05,...,1,1.0,1,2,-1.0,0.0,2.0,18.0,123.0,141.0
3,6,8,15.0,Dice+Focal,16.0,gda,632,False,mix,0.05,...,1,-1.0,1,2,1.0,0.0,2.0,755.0,20.0,775.0
4,6,8,15.0,Dice+Focal,16.0,gda,632,False,mix,0.05,...,1,-1.0,1,2,1.0,0.0,2.0,755.0,20.0,775.0


In [291]:
df = df123
df['comb_key'] = df['tr_val_x'] + '|' + \
                df['tr_val_y'] + '|' + \
                df['tr_val']
df.head()

,warmup_epochs_x,batch_size_x,1_epoch,loss_x,epochs_x,val_x,Runtime_x,ema_x,sub_x,wd_x,...,do_t,re_v,real,do_v,dom,dist,w_t,w_v,workload,comb_key
0,6,8,15.0,Dice+Focal,16.0,gda,632,False,mix,0.05,...,1.0,1,2,-1.0,0.0,2.0,18.0,123.0,141.0,sub_gda|subm_m|gda_dk
1,6,8,15.0,Dice+Focal,16.0,gda,632,False,mix,0.05,...,1.0,1,2,-1.0,0.0,2.0,18.0,123.0,141.0,sub_gda|subm_m|gda_dk
2,6,8,15.0,Dice+Focal,16.0,gda,632,False,mix,0.05,...,1.0,1,2,-1.0,0.0,2.0,18.0,123.0,141.0,sub_gda|subm_m|gda_dk
3,6,8,15.0,Dice+Focal,16.0,gda,632,False,mix,0.05,...,-1.0,1,2,1.0,0.0,2.0,755.0,20.0,775.0,sub_gda|subm_m|dk_gda
4,6,8,15.0,Dice+Focal,16.0,gda,632,False,mix,0.05,...,-1.0,1,2,1.0,0.0,2.0,755.0,20.0,775.0,sub_gda|subm_m|dk_gda


In [292]:
df123.columns

Index(['warmup_epochs_x', 'batch_size_x', '1_epoch', 'loss_x', 'epochs_x',
       'val_x', 'Runtime_x', 'ema_x', 'sub_x', 'wd_x', 'lrenc_x', 'Sweep_x',
       'lrdec_x', '1_test_DK_iou', '1_test_SYNT_iou', '1_test_GDA_iou',
       'train_x', 'tr_val_x', 'src_x', 'fn_x', 'sub_mult_x', 're_t_x',
       'do_t_x', 're_v_x', 'real_x', 'do_v_x', 'dom_x', 'dist_x', 'w_t_x',
       'w_v_x', 'workload_x', 'ID_x', '2_epoch', 'warmup_epochs_y', 'mod_ph1',
       'batch_size_y', 'loss_y', 'ID_y', 'epochs_y', 'val_y', 'Runtime_y',
       'ema_y', 'sub_y', 'wd_y', 'lrenc_y', 'Sweep_y', 'lrdec_y',
       '2_test_GDA_iou', '2_test_SYNT_iou', '2_test_DK_iou', 'train_y',
       'tr_val_y', 'src_y', 'fn_y', 'sub_mult_y', 're_t_y', 'do_t_y', 're_v_y',
       'real_y', 'do_v_y', 'dom_y', 'dist_y', 'w_t_y', 'w_v_y', 'workload_y',
       'val', 'wd', 'lrenc', 'epochs', 'Runtime', 'loss', 'Sweep', 'lrdec',
       'warmup_epochs', 'batch_size', 'ID', 'mod_ph2', 'ema', '3_epoch',
       '3_test_DK_iou', '3_test

In [293]:
print(ph1.size)
print(df12.size)
print(df123.size)

29984
66885
94472


In [294]:
print(cnc.size)

123060


In [295]:
df.to_csv(f'{FDIR}/ph123b.csv') # OK